# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Lane: Refresh / Content Opportunity Scoring (provisional). I'm picking this one over the other three because its output is something a person can actually act on -- a ranked list of specific pages to look at first, with a reason attached to each one, not just a general pattern. There's already a concrete reason to think this is worth 7 weeks: on this same 30,000-page starter slice, a hand-written rule baseline only gets 12 of its top 50 picks right, while a random forest ranking the same pages gets 37 of 50 right (see outputs/model_report.md). That's real evidence a learned ranking can beat FlyRank's existing rule here, not just a hope. I'm keeping "freestyle" in my back pocket -- if the reason codes turn out to be too thin once I dig into the warehouse data, Ranking Signal Analysis is my fallback.

In [2]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"
(not os.path.isdir(REPO_DIR)) and IN_COLAB and subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
IN_COLAB and os.chdir(REPO_DIR)
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- check you're at the repo root"
print("Working dir:", os.getcwd())
lanes = {"Ranking Signal Analysis": "signal report + evidence-backed recommendations", "Refresh / Content Opportunity Scoring": "ranked review queue with scores, actions, reason codes", "Structured Content Archetype Clustering": "cluster profiles + action mapping", "CTR / Engagement Opportunity Scoring": "ranked opportunity score, position/volume adjusted"}
my_lane = "Refresh / Content Opportunity Scoring"
assert my_lane in lanes, "not a documented lane -- check docs/ml-intern-dataset-and-lane-guide.md"
print("Lane picked:", my_lane, "-- output:", lanes[my_lane])

Working dir: /content/flyrank-ml-internship-starter
Lane picked: Refresh / Content Opportunity Scoring -- output: ranked review queue with scores, actions, reason codes


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision: out of thousands of existing pages, which one should a content reviewer look at first this cycle? Who acts: a FlyRank content/SEO reviewer with limited review hours per week -- they can't check every page by hand, so the output has to be a ranked queue, not just a flag. Cost of a wrong call: two different mistakes, two different costs. A false positive (flagging a page that didn't really need attention) wastes a reviewer's scarce hours. A false negative (missing a page that really is declining with real demand) means that page keeps quietly losing traffic until someone happens to notice on their own. Because reviewer time is the constrained resource, the ranking needs to be trustworthy near the top of the queue especially -- that's where Precision@K becomes the metric that matters, more than overall accuracy.

In [3]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
demand_floor = 100
declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= demand_floor)]
print("Total pages in this slice:", len(df))
print("Declining pages with real demand (impressions_90d >=", demand_floor, "):", len(declining_with_demand))
print("Share of all pages:", round(len(declining_with_demand) / len(df) * 100, 1), "%")
print("No reviewer works through that many pages by hand each cycle -- ranking WHICH ONE to fix first is the real decision.")

Total pages in this slice: 30000
Declining pages with real demand (impressions_90d >= 100 ): 13152
Share of all pages: 43.8 %
No reviewer works through that many pages by hand each cycle -- ranking WHICH ONE to fix first is the real decision.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

On the 30,000-page starter slice (32 pseudonymized clients): 1) 13,152 pages (43.8%) are declining AND still have real search demand (impressions_90d >= 100) -- far too many for any reviewer to work through by hand, which is exactly the prioritization problem this lane targets. 2) A hand-written rule baseline correctly flags only 12 of its top 50 picks (Precision@50 = 0.240); a random forest ranking the same pages gets 37 of 50 right (Precision@50 = 0.740), about 3x better on the same data -- real evidence that a learned ranking earns its place here. 3) Even the pipeline's own high-confidence tier alone (3,605 pages) is far more than one reviewer clears in a week, which is why ranking WHICH page to look at first matters more than just flagging pages as "at risk."

In [4]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
print("1) Pages:", len(df), "| Clients:", df["client_id"].nunique())
print("   Declining pages with real demand:", len(declining_with_demand), "=", round(len(declining_with_demand) / len(df) * 100, 1), "% of all pages")
report_lines = open("outputs/model_report.md").read().splitlines()
base_line = next(l for l in report_lines if l.startswith("| baseline_rules"))
rf_line = next(l for l in report_lines if l.startswith("| random_forest"))
base_p50 = float(base_line.split("|")[4].strip())
rf_p50 = float(rf_line.split("|")[4].strip())
print("2) Baseline Precision@50:", base_p50, "vs Random forest Precision@50:", rf_p50, "=", round(rf_p50 / base_p50, 1), "x better")
high_conf_line = next(l for l in report_lines if l.startswith("- High-confidence items:"))
print("3)", high_conf_line.strip(), "-- already more than a reviewer clears in a week")

1) Pages: 30000 | Clients: 32
   Declining pages with real demand: 13152 = 43.8 % of all pages
2) Baseline Precision@50: 0.24 vs Random forest Precision@50: 0.74 = 3.1 x better
3) - High-confidence items: 3,605 -- already more than a reviewer clears in a week


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I can say: this work observes real (anonymized) search and engagement signals over a trailing 90-day window, and it can measure -- with client-holdout validation -- whether a learned ranking beats a fixed rule using Precision@K. Any output is decision-support: it tells a reviewer which page to look at first, based on observed and directional patterns, never a guarantee. What I can't say: I can't claim that refreshing a page CAUSES it to recover -- that needs a controlled experiment (before/after with a control group), which this data alone doesn't give me. I also won't claim to be predicting Google's ranking algorithm -- I'm only using observable signals FlyRank already measures (impressions, clicks, position, engagement), never the product's own decision flags (health_score, priority_score) as model inputs, and trend_direction/trend_pct stay label-only, never a feature. Anything I report stays "observed" or "directional," not proof.

In [5]:
label_source_columns = {"trend_direction", "trend_pct"}
product_decision_columns = {"health_score", "priority_score", "action_type"}
overlap_with_product_flags = set(df.columns) & product_decision_columns
print("Label-source columns held out of any feature list, never inputs:", sorted(label_source_columns))
print("Product decision columns present in this dataset, should be none:", sorted(overlap_with_product_flags))
print("No experiment ran on this data, so nothing here will claim a refresh CAUSES recovery -- only that it ranks review candidates.")

Label-source columns held out of any feature list, never inputs: ['trend_direction', 'trend_pct']
Product decision columns present in this dataset, should be none: []
No experiment ran on this data, so nothing here will claim a refresh CAUSES recovery -- only that it ranks review candidates.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.